## Step 1: Imports and set up base paths and output path

In [ ]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_origin
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from rasterio.warp import reproject
from rasterio.enums import Resampling
import connectivity
import tifffile
import rioxarray as rxr
import os
from osgeo import gdal
import rioxarray
import Robynlibrary as Robyn
import Robyn_forest_classes
from rasterio.windows import from_bounds
import matplotlib.pyplot as plt

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import rasterio, numpy as np
import geopandas as gpd
from pathlib import Path
import matplotlib.colors as mcolors


#### Define base paths and set inputs

In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis")
inputs_dir    = base_path / "Inputs"
output_dir = base_path / "Outputs"
processed_dir = base_path / "Processed_data"


In [ ]:
land_use_path = inputs_dir / "2013_landuse_LandCover.shp"
terrestrial_landcover = gpd.read_file(land_use_path).copy() # Optional: if you want to preserve the original
terrestrial_landcover.crs

In [ ]:
jamaica_boundary_path = base_path / "Inputs/Boundaries/jamaica.gpkg"
jamaica_boundary = gpd.read_file(jamaica_boundary_path)
print(jamaica_boundary.crs)

### Catchments 

In [ ]:
catchments = base_path / "major_basins_plus_coastal.gpkg"
catchments = gpd.read_file(catchments)

In [ ]:
catchments.head()

## Step 2: Read in condition spreadsheets and merge them with Jamaica landcover file 

In [ ]:
# Read your condition mapping spreadsheet
# mapping_table = pd.read_excel(base_path / "landcover_condition_mapping.xlsx")
test_condition_mapping_ones = pd.read_excel(base_path / "jamaica_landcover_condition_test_ones.xlsx")
test_condition_mapping_zeros = pd.read_excel(base_path / "jamaica_landcover_condition_test_zeros.xlsx")
# afforestable_condition_codes = pd.read_excel(base_path / "jamaica_afforestable_condition.xlsx")
baseline_condition_mixed_landcover_50_50_power4 = pd.read_excel(base_path / "landcover_condition_mapping_power4_50_50_mixed.xlsx")
afforestable_condition_power4_mixed_landcover_50_50_power4 = pd.read_excel(base_path / "jamaica_afforestable_condition_50_50_mixed_power4.xlsx")

In [ ]:
# Merge the mapping with your geodataframe on the 'Classify' column
# landcover_with_condition_baseline = terrestrial_landcover.merge(mapping_table, on='Classify')
landcover_with_condition_ones = terrestrial_landcover.merge(test_condition_mapping_ones, on='Classify')
landcover_with_condition_zeros = terrestrial_landcover.merge(test_condition_mapping_zeros, on='Classify')
# landcover_with_condition_afforestable = terrestrial_landcover.merge(afforestable_condition_codes, on='Classify')
landcover_with_condition_baseline_condition_mixed_landcover_50_50_power4 = terrestrial_landcover.merge(baseline_condition_mixed_landcover_50_50_power4, on='Classify')
landcover_with_condition_afforestable_condition_power4_mixed_landcover_50_50_power4 = terrestrial_landcover.merge(afforestable_condition_power4_mixed_landcover_50_50_power4, on='Classify')

## Step 3: Convert land use condition files to rasters

#### Define output files paths for the condition rasters 

In [ ]:
# Use the function for each condition and capture the outputs
# baseline_out = output_dir / "landcover_condition.tif"
ones_out = output_dir / "landcover_condition_test_ones.tif"
zeros_out = output_dir / "landcover_condition_test_zeros.tif"
# afforestable_out = output_dir / "landcover_condition_afforestable.tif"
baseline_condition_mixed_landcover_50_50_power4_out = output_dir / "baseline_landcover_condition_mixed_landcover_50_50_power4.tif"
afforestable_condition_power4_mixed_landcover_50_50_power4_out = output_dir / "afforestable_landcover_condition_50_50_power4.tif"

#### Rasterize using the Robyn.rasterize_condition function 

In [ ]:
# condition_raster_baseline, transform = Robyn.rasterize_condition(landcover_with_condition_baseline, baseline_out)
# condition_raster_ones, _ = Robyn.rasterize_condition(landcover_with_condition_ones, ones_out)

condition_raster_ones, transform = Robyn.rasterize_condition(
    landcover_with_condition_ones,
    ones_out
)

condition_raster_zeros, _ = Robyn.rasterize_condition(landcover_with_condition_zeros, zeros_out)
# condition_raster_afforestable, _ = Robyn.rasterize_condition(landcover_with_condition_afforestable, afforestable_out)
condition_raster_baseline_condition_mixed_landcover_50_50_power4, _ = Robyn.rasterize_condition(
    landcover_with_condition_baseline_condition_mixed_landcover_50_50_power4,
    baseline_condition_mixed_landcover_50_50_power4_out
)

afforestable_condition_raster_power4_mixed_landcover_50_50_power4, _ = Robyn.rasterize_condition(
    landcover_with_condition_afforestable_condition_power4_mixed_landcover_50_50_power4,
    afforestable_condition_power4_mixed_landcover_50_50_power4_out
)

## Step 4: resample to 100 by 100 size cells rather than the current 10 by 10 

In [ ]:
# 1) Compute the common spatial extent from your baseline GeoDataFrame
bounds = landcover_with_condition_ones.total_bounds

# 2) Define output file names for each resampled raster
ones_resampled_file = output_dir / "landcover_condition_test_ones_resampled.tif"
zeros_resampled_file = output_dir / "landcover_condition_test_zeros_resampled.tif"

baseline_50_50_power4_resampled_file = (
    output_dir / "baseline_landcover_condition_mixed_landcover_50_50_power4_resampled.tif"
)

afforestable_50_50_power4_resampled_file = (
    output_dir / "afforestable_landcover_condition_50_50_power4_resampled.tif"
)

# # 3) Resample each raster into that common extent & CRS
# Robyn.resample_and_save(
#     condition_raster_baseline,
#     transform,
#     landcover_with_condition_baseline.crs,
#     bounds,
#     baseline_resampled_file
# )

Robyn.resample_and_save(
    condition_raster_ones,
    transform,
    landcover_with_condition_ones.crs,
    bounds,
    ones_resampled_file
)

Robyn.resample_and_save(
    condition_raster_zeros,
    transform,
    landcover_with_condition_zeros.crs,
    bounds,
    zeros_resampled_file
)


Robyn.resample_and_save(
    condition_raster_baseline_condition_mixed_landcover_50_50_power4,
    transform,
    landcover_with_condition_baseline_condition_mixed_landcover_50_50_power4.crs,
    bounds,
    baseline_50_50_power4_resampled_file
)


Robyn.resample_and_save(
    afforestable_condition_raster_power4_mixed_landcover_50_50_power4,
    transform,
    landcover_with_condition_afforestable_condition_power4_mixed_landcover_50_50_power4.crs,
    bounds,
    afforestable_50_50_power4_resampled_file
)

### Step 4: resample to 100 by 100 size cells rather than the current 10 by 10 

In [ ]:
# 1) Compute the common spatial extent from your baseline GeoDataFrame
bounds = landcover_with_condition_ones.total_bounds

# 2) Define output file names for each resampled raster
ones_resampled_file = output_dir / "landcover_condition_test_ones_resampled.tif"
zeros_resampled_file = output_dir / "landcover_condition_test_zeros_resampled.tif"


baseline_50_50_power4_resampled_file = (
    output_dir / "baseline_landcover_condition_mixed_landcover_50_50_power4_resampled.tif"
)

afforestable_50_50_power4_resampled_file = (
    output_dir / "afforestable_landcover_condition_50_50_power4_resampled.tif"
)

# # 3) Resample each raster into that common extent & CRS
# Robyn.resample_and_save(
#     condition_raster_baseline,
#     transform,
#     landcover_with_condition_baseline.crs,
#     bounds,
#     baseline_resampled_file
# )

Robyn.resample_and_save(
    condition_raster_ones,
    transform,
    landcover_with_condition_ones.crs,
    bounds,
    ones_resampled_file
)

Robyn.resample_and_save(
    condition_raster_zeros,
    transform,
    landcover_with_condition_zeros.crs,
    bounds,
    zeros_resampled_file
)


Robyn.resample_and_save(
    condition_raster_baseline_condition_mixed_landcover_50_50_power4,
    transform,
    landcover_with_condition_baseline_condition_mixed_landcover_50_50_power4.crs,
    bounds,
    baseline_50_50_power4_resampled_file
)


Robyn.resample_and_save(
    afforestable_condition_raster_power4_mixed_landcover_50_50_power4,
    transform,
    landcover_with_condition_afforestable_condition_power4_mixed_landcover_50_50_power4.crs,
    bounds,
    afforestable_50_50_power4_resampled_file
)

## Step 5: Connectivity analysis 

In [ ]:
# # Compute connectivity using the already-defined resampled-file paths:
# # baseline_connectivity = Robyn.compute_connectivity(baseline_resampled_file)
ones_connectivity     = Robyn.compute_connectivity(ones_resampled_file)
zeros_connectivity    = Robyn.compute_connectivity(zeros_resampled_file)
baseline_50_50_conn   = Robyn.compute_connectivity(baseline_50_50_power4_resampled_file)
afforest_50_50_conn   = Robyn.compute_connectivity(afforestable_50_50_power4_resampled_file)

# Print out the results
# print(f"Baseline connectivity: {baseline_connectivity}")
print(f"Ones connectivity: {ones_connectivity}")
print(f"Zeros connectivity: {zeros_connectivity}")
print(f"Baseline 50/50 power4 connectivity: {baseline_50_50_conn}")
print(f"Afforestable 50/50 power4 connectivity: {afforest_50_50_conn}")

In [ ]:
# Compute normalized connectivity (compared to the ones/zeros extremes) for each scenario
# baseline_normalized_extremes       = Robyn.calc_connectivity_normalized(
#     baseline_connectivity,
#     ones_connectivity,
#     zeros_connectivity
# )

baseline_50_50_normalized_extremes = Robyn.calc_connectivity_normalized(
    baseline_50_50_conn,
    ones_connectivity,
    zeros_connectivity
)


afforest_50_50_normalized_extremes = Robyn.calc_connectivity_normalized(
    afforest_50_50_conn,
    ones_connectivity,
    zeros_connectivity
)

# Print them out
# print(f"Baseline normalized (compared to extremes):                       {baseline_normalized_extremes:.2f}%")
print(f"Baseline 50/50 power4 normalized (compared to extremes):         {baseline_50_50_normalized_extremes:.2f}%")
print(f"Afforestable 50/50 power4 normalized (compared to extremes):     {afforest_50_50_normalized_extremes:.2f}%")

## Step 6: calculate forest area as a proportion of Jamaica land cover area 

In [ ]:
# 1) total land area
total_land_area = terrestrial_landcover.geometry.area.sum()

# 2) your two class‐sets
flood_classes = Robyn_forest_classes.forest_flood_equivalent_classes
afforest_classes = Robyn_forest_classes.afforestable_classes_including_agricultural

# 4) build the union for the future scenario
future_classes = flood_classes.union(afforest_classes)

# 5) flatten your mixed‐fractions for the future scenario
mixed = Robyn_forest_classes.mixed_land_use_fractions
future_mixed = {
    cls: fracs['afforestable_including_agriculture']
    for cls, fracs in mixed.items()
    if 'afforestable_including_agriculture' in fracs
}

# 6) compute areas (m²)
baseline_area = Robyn.compute_forest_area(
    terrestrial_landcover,
    flood_classes,
    mixed={cls: fracs['forest_flood_equivalent_classes']
           for cls, fracs in mixed.items()
           if 'forest_flood_equivalent_classes' in fracs}
)

future_area   = Robyn.compute_forest_area(
    terrestrial_landcover,
    future_classes,
    mixed=future_mixed
)

# 7) percentages & km²
baseline_pct = baseline_area / total_land_area * 100
future_pct   = future_area   / total_land_area * 100

baseline_km2 = baseline_area / 1e6
future_km2   = future_area   / 1e6
total_km2    = total_land_area / 1e6

print(f"Total land: {total_land_area:,.0f} m² ({total_km2:.2f} km²)")
print(f"Baseline forest: {baseline_area:,.0f} m² ({baseline_km2:.2f} km²) → {baseline_pct:.2f}%")
print(f"Future forest:   {future_area:,.0f} m² ({future_km2:.2f} km²) → {future_pct:.2f}%")

## Step 7: Catchment-level analysis 

In [ ]:
# Calculate the area in m² and km², and add them as new columns
catchments["area_m2"] = catchments.geometry.area
catchments["area_km2"] = catchments["area_m2"] / 1e6

# Add a new column with a unique new ID starting at 1
catchments["new_id"] = range(1, len(catchments) + 1)

# Calculate the equivalent diameter (distance across) in meters
# Equivalent diameter = 2 * sqrt(area_m2 / pi)
catchments["equiv_diam_m"] = 2 * np.sqrt(catchments["area_m2"] / np.pi)


# Calculate the smallest and largest HYBAS_ID area (in m² and km²)
smallest_area_m2 = catchments["area_m2"].min()
mean_area_m2 = catchments["area_m2"].mean()
largest_area_m2 = catchments["area_m2"].max()
smallest_area_km2 = catchments["area_km2"].min()
mean_area_km2 = catchments["area_km2"].mean()
largest_area_km2 = catchments["area_km2"].max()

smallest_diam = catchments["equiv_diam_m"].min()
mean_diam = catchments["equiv_diam_m"].mean()
largest_diam = catchments["equiv_diam_m"].max()


# Print summary statistics of hydrobasins

print("Smallest catchments area (m²):", smallest_area_m2)
print("Mean catchments area (m²):", mean_area_m2)
print("Largest catchments area (m²):", largest_area_m2)
print("Smallest catchments area (km²):", smallest_area_km2)
print("Mean catchments area (km²):", mean_area_km2)
print("Largest catchments area (km²):", largest_area_km2)

print("Smallest equivalent diameter (m):", smallest_diam)
print("Mean equivalent diameter (m):", mean_diam)
print("Largest equivalent diameter (m):", largest_diam)
number_catchments = catchments["new_id"].max()
print(number_catchments)


# # Write the modified hydrobasins to a new shapefile
# catchments.to_file("catchments_modified.shp")
catchments.to_file("catchments_modified.gpkg", driver="GPKG")


catchments.head()

In [ ]:
# Define the raster resolution (in meters) and extent
pixel_size = 100  # Change this value to your desired resolution (e.g., 10m)
minx, miny, maxx, maxy = catchments.total_bounds

# Compute width and height in terms of pixels
width = int(np.ceil((maxx - minx) / pixel_size))
height = int(np.ceil((maxy - miny) / pixel_size))

# Create an affine transform for the raster (origin at top-left)
transform = from_origin(minx, maxy, pixel_size, pixel_size)

# Create (geometry, value) pairs for rasterization using the "new_id" column as the value.
shapes = ((geom, value) for geom, value in zip(catchments.geometry, catchments['new_id']))

# Rasterize the geometries into a NumPy array. 
raster = rasterize(
    shapes=shapes,
    out_shape=(height, width),
    transform=transform,
    fill=0,  # Pixels that don't fall within any geometry will be assigned the "fill" value (0).
    dtype=np.uint16  # Change type as needed based on your data range
)

In [ ]:
# --- raster template: pick one you already saved in Outputs ---
template_raster_candidates = [
    output_dir / "baseline_landcover_condition_mixed_landcover_50_50_power4.tif",
    output_dir / "afforestable_landcover_condition_50_50_power4.tif",
    output_dir / "landcover_condition_test_ones.tif",
    output_dir / "landcover_condition_test_zeros.tif",
]
template_raster_path = next((p for p in template_raster_candidates if p.exists()), None)
if template_raster_path is None:
    # fallback: grab any .tif in Outputs
    any_tifs = sorted(output_dir.glob("*.tif"))
    if not any_tifs:
        raise FileNotFoundError(f"No template raster found in {output_dir}")
    template_raster_path = any_tifs[0]

# --- outputs to write later ---
catchments_raster_path = processed_dir / "catchments_raster.tif"
catchments_id_map_path = processed_dir / "catchments_id_map.parquet"

# --- quick sanity echo (optional) ---
print("Template raster:", template_raster_path)
print("Catchments path:", catchments)
print("Will write raster:", catchments_raster_path)
print("Will write ID map:", catchments_id_map_path)

In [ ]:
baseline_power4_resampled_file = template_raster_path

if not catchments_raster_path.exists():
    with rasterio.open(template_raster_path) as tmpl:
        tmpl_crs       = tmpl.crs
        tmpl_transform = tmpl.transform
        tmpl_shape     = (tmpl.height, tmpl.width)
        tmpl_profile   = tmpl.profile

    # use your existing GeoDataFrame 'catchments'
    gdf = catchments if catchments.crs == tmpl_crs else catchments.to_crs(tmpl_crs)

    # ensure a small contiguous integer ID
    if "catchment_uid" not in gdf.columns:
        gdf = gdf.reset_index(drop=True).copy()
        gdf["catchment_uid"] = np.arange(1, len(gdf) + 1, dtype=np.uint32)

    burn = rasterio.features.rasterize(
        ((geom, int(v)) for geom, v in zip(gdf.geometry, gdf["catchment_uid"])),
        out_shape=tmpl_shape,
        transform=tmpl_transform,
        fill=0,
        dtype="uint32",
        all_touched=False,
    )

    profile = tmpl_profile.copy()
    profile.update(count=1, dtype=burn.dtype, nodata=0, compress="deflate", predictor=2, tiled=True)
    with rasterio.open(catchments_raster_path, "w", **profile) as dst:
        dst.write(burn, 1)

    print("Wrote catchments raster →", catchments_raster_path)

In [ ]:
with rasterio.open(catchments_raster_path) as src:
    transform = src.transform
    bounds    = src.bounds
    width, height = src.width, src.height
    print("Catchments raster:")
    print("  CRS:",             src.crs)
    print("  Width × Height:",  width, "×", height)
    print("  Pixel size:",      transform.a, "×", transform.e)
    print("  Bounds:",          bounds)

with rasterio.open(baseline_power4_resampled_file) as tmpl: 
    print("Template landcover raster:")
    print("  CRS:", tmpl.crs)
    print("  Width × Height:", tmpl.width, "×", tmpl.height)
    print("  Pixel size:",     tmpl.transform.a, "×", tmpl.transform.e)
    print("  Bounds:",         tmpl.bounds)


In [ ]:
# Use the template you picked earlier
baseline_power4_resampled_file = template_raster_path

# Pick your afforested/alternative condition raster in Outputs
afforestable_power4_resampled_file = next(p for p in [
    output_dir / "afforestable_landcover_condition_50_50_power4.tif",
    output_dir / "landcover_condition_test_ones.tif"  # fallback if needed
] if p.exists())

# Load arrays (swap in catchments_raster_path; no more 'clipped_catchments_raster')
catchment_raster_upd = Robyn.open_raster_as_array(str(catchments_raster_path))
baseline_condition_raster_power4_upd = Robyn.open_raster_as_array(str(baseline_power4_resampled_file))
afforested_condition_raster_power4_upd = Robyn.open_raster_as_array(str(afforestable_power4_resampled_file))

# If your helper returns (1, H, W), squeeze to (H, W)
catchment_raster_upd = np.squeeze(catchment_raster_upd)
baseline_condition_raster_power4_upd = np.squeeze(baseline_condition_raster_power4_upd)
afforested_condition_raster_power4_upd = np.squeeze(afforested_condition_raster_power4_upd)

print("Unique catchment IDs:", np.unique(catchment_raster_upd))
print("Catchment raster shape:",  catchment_raster_upd.shape)
print("Baseline raster shape:",   baseline_condition_raster_power4_upd.shape)
print("Afforested raster shape:", afforested_condition_raster_power4_upd.shape)

# Quick sanity: all rasters must align
assert catchment_raster_upd.shape == baseline_condition_raster_power4_upd.shape == afforested_condition_raster_power4_upd.shape

In [ ]:
# # ---- minimal prep (keep this before the loop) ----
# # arrays you already loaded earlier:
# #   catchment_raster_upd
# #   baseline_condition_raster_power4_upd
# #   afforested_condition_raster_power4_upd

# zeros_resampled_file = output_dir / "landcover_condition_test_zeros.tif"
# ones_resampled_file  = output_dir / "landcover_condition_test_ones.tif"

# n_processes = os.cpu_count()
# generations_mode = "one_generation"
# number_of_species_generations = 1

# # ---- power-4 only outputs ----
# out_cols = [
#     "baseline_catchment_connectivity_power4",
#     "baseline_normalized_power4",
#     "afforested_catchment_connectivity_power4",
#     "percentage_difference_catchment_connectivity_power4",
#     "afforested_normalized_power4",
# ]
# for c in out_cols:
#     if c not in catchments.columns:
#         catchments[c] = np.nan

# # optional: remove old non-power4 columns if present
# for c in ["baseline_catchment_connectivity","baseline_normalized",
#           "afforested_catchment_connectivity","percentage_difference_catchment_connectivity",
#           "afforested_normalized_extremes"]:
#     if c in catchments.columns:
#         catchments.drop(columns=c, inplace=True)

# # ---------- PRINT PIXEL COUNTS (fast) ----------
# number_catchments = int(catchment_raster_upd.max())

# # one pass over the raster to get counts for all IDs
# counts = np.bincount(
#     catchment_raster_upd.ravel().astype(np.int64),
#     minlength=number_catchments + 1
# )

# # print in numeric order (like before)
# for cid in range(1, number_catchments + 1):
#     pixel_count = int(counts[cid])
#     print(f"Catchment {cid}: non-zero pixels = {pixel_count}", flush=True)

# # ---------- ORDER TO PROCESS ----------
# MIN_PIXELS = 0           # raise to skip tiny slivers during testing
# SMALL_FIRST = False      # False = numeric order (1..N), True = small→large

# ids_with_pixels = [cid for cid in range(1, number_catchments + 1) if counts[cid] >= MIN_PIXELS]
# order = (sorted(ids_with_pixels, key=lambda cid: counts[cid]) if SMALL_FIRST else ids_with_pixels)

# # Helper
# def _pct_diff(new, base):
#     return np.nan if base == 0 else (new - base) / base * 100.0

# def _norm_safe(baseline, maxv, minv, eps=1e-12):   # NEW: avoids ZeroDivisionError
#     denom = maxv - minv
#     if np.isfinite(denom) and abs(denom) > eps:
#         return ((baseline - minv) / denom) * 100.0
#     return np.nan

# # ---------- attach catchment_uid from the raster (so writes line up) ----------
# with rasterio.open(catchments_raster_path) as src:
#     r_crs = src.crs
# if catchments.crs != r_crs:
#     catchments = catchments.to_crs(r_crs)

# pts = catchments.geometry.representative_point()
# coords = [(p.x, p.y) for p in pts]
# with rasterio.open(catchments_raster_path) as src:
#     catchments_ids_from_raster = [int(v[0]) for v in src.sample(coords)]
# catchments["catchment_uid"] = np.array(catchments_ids_from_raster, dtype=np.uint32)

# n0 = int((catchments["catchment_uid"] == 0).sum())
# if n0:
#     print(f"Warning: {n0} polygons sampled ID 0 (background). They may be smaller than a pixel.")

# # ---------- POWER-4 CONNECTIVITY (print per catchment) ----------
# with rasterio.open(zeros_resampled_file) as rz, rasterio.open(ones_resampled_file) as ro:
#     for lambda_parameter in [5]:
#         done, total = 0, len(order)
#         for cid in order:
#             # build a tight crop window lazily for this ID only
#             rows, cols = np.where(catchment_raster_upd == cid)
#             if rows.size == 0:
#                 continue
#             r0, r1 = rows.min(), rows.max() + 1
#             c0, c1 = cols.min(), cols.max() + 1
#             sl_r, sl_c = slice(r0, r1), slice(c0, c1)

#             print(f"→ λ={lambda_parameter} starting cid={cid} (px={counts[cid]}, "
#                   f"win={r1-r0}×{c1-c0})", flush=True)

#             mask = (catchment_raster_upd[sl_r, sl_c] == cid).astype(np.uint8, copy=False)
#             if mask.sum() == 0:
#                 continue

#             # crop arrays already in memory
#             base = baseline_condition_raster_power4_upd[sl_r, sl_c]
#             aff  = afforested_condition_raster_power4_upd[sl_r, sl_c]

#             # lazily read just this window from zeros/ones on disk
#             win = rasterio.windows.Window.from_slices(sl_r, sl_c)
#             zeros_ = rz.read(1, window=win)
#             ones_  = ro.read(1, window=win)

#             baseline_conn_power4 = connectivity.landscape_connectivity(
#                 base, n_processes, mask, lambda_parameter, generations_mode, number_of_species_generations
#             )
#             aff_conn_power4 = connectivity.landscape_connectivity(
#                 aff,  n_processes, mask, lambda_parameter, generations_mode, number_of_species_generations
#             )
#             test_zeros_conn = connectivity.landscape_connectivity(
#                 zeros_, n_processes, mask, lambda_parameter, generations_mode, number_of_species_generations
#             )
#             test_ones_conn  = connectivity.landscape_connectivity(
#                 ones_,  n_processes, mask, lambda_parameter, generations_mode, number_of_species_generations
#             )

#             perc_diff_power4 = _pct_diff(aff_conn_power4, baseline_conn_power4)
#             baseline_norm_p4 = Robyn.calc_connectivity_normalized(baseline_conn_power4, test_ones_conn, test_zeros_conn)
#             aff_norm_p4      = Robyn.calc_connectivity_normalized(aff_conn_power4,      test_ones_conn, test_zeros_conn)

#             # print per-catchment connectivity line
#             print(
#                 f"[λ={lambda_parameter}] Catchment {cid}: "
#                 f"baseline_p4={baseline_conn_power4:.2f}, baseline_norm_p4={baseline_norm_p4:.3f}, "
#                 f"afforested_p4={aff_conn_power4:.2f}, diff_p4={perc_diff_power4:.1f}%, "
#                 f"aff_norm_p4={aff_norm_p4:.3f}",
#                 flush=True
#             )

#             # write back to GeoDataFrame
#             catchments.loc[catchments["catchment_uid"] == cid, out_cols] = [
#                 baseline_conn_power4, baseline_norm_p4, aff_conn_power4, perc_diff_power4, aff_norm_p4
#             ]

#             done += 1
#             if (done % 5) == 0 or done == total:
#                 print(f"Progress λ={lambda_parameter}: {done}/{total} (cid={cid}, px={counts[cid]})", flush=True)

#         # Save one CSV per λ
#         out_csv = output_dir / f"lambda_{lambda_parameter}_catchment_connectivity.csv"
#         catchments.drop(columns=["geometry"], errors="ignore").to_csv(out_csv, index=False)
#         print(f"Done with λ={lambda_parameter}. Wrote {out_csv}")

In [ ]:
# ---- minimal prep (keep this before the loop) ----
# arrays you already loaded earlier:
#   catchment_raster_upd
#   baseline_condition_raster_power4_upd
#   afforested_condition_raster_power4_upd


zeros_resampled_file = output_dir / "landcover_condition_test_zeros.tif"
ones_resampled_file  = output_dir / "landcover_condition_test_ones.tif"

n_processes = os.cpu_count()
generations_mode = "one_generation"
number_of_species_generations = 1

# ---- power-4 only outputs ----
out_cols = [
    "baseline_catchment_connectivity_power4",
    "baseline_normalized_power4",
    "afforested_catchment_connectivity_power4",
    "percentage_difference_catchment_connectivity_power4",
    "afforested_normalized_power4",
]
for c in out_cols:
    if c not in catchments.columns:
        catchments[c] = np.nan

# optional: remove old non-power4 columns if present
for c in ["baseline_catchment_connectivity","baseline_normalized",
          "afforested_catchment_connectivity","percentage_difference_catchment_connectivity",
          "afforested_normalized_extremes"]:
    if c in catchments.columns:
        catchments.drop(columns=c, inplace=True)

# ---------- PRINT PIXEL COUNTS (fast) ----------
number_catchments = int(catchment_raster_upd.max())

# one pass over the raster to get counts for all IDs
counts = np.bincount(
    catchment_raster_upd.ravel().astype(np.int64),
    minlength=number_catchments + 1
)

# print in numeric order (like before)
for cid in range(1, number_catchments + 1):
    pixel_count = int(counts[cid])
    print(f"Catchment {cid}: non-zero pixels = {pixel_count}", flush=True)

# ---------- ORDER TO PROCESS ----------
MIN_PIXELS = 0           # raise to skip tiny slivers during testing
SMALL_FIRST = False      # False = numeric order (1..N), True = small→large

ids_with_pixels = [cid for cid in range(1, number_catchments + 1) if counts[cid] >= MIN_PIXELS]
order = (sorted(ids_with_pixels, key=lambda cid: counts[cid]) if SMALL_FIRST else ids_with_pixels)

# Helpers
def _pct_diff(new, base):
    return np.nan if base == 0 else (new - base) / base * 100.0

def _norm_try(baseline, maxv, minv):
    # Wrap your Robyn function to avoid ZeroDivisionError on tiny windows
    try:
        return Robyn.calc_connectivity_normalized(baseline, maxv, minv)
    except ZeroDivisionError:
        return np.nan

# ---------- attach catchment_uid from the raster (so writes line up) ----------
with rasterio.open(catchments_raster_path) as src:
    r_crs = src.crs
if catchments.crs != r_crs:
    catchments = catchments.to_crs(r_crs)

pts = catchments.geometry.representative_point()
coords = [(p.x, p.y) for p in pts]
with rasterio.open(catchments_raster_path) as src:
    catchments_ids_from_raster = [int(v[0]) for v in src.sample(coords)]
catchments["catchment_uid"] = np.array(catchments_ids_from_raster, dtype=np.uint32)

n0 = int((catchments["catchment_uid"] == 0).sum())
if n0:
    print(f"Warning: {n0} polygons sampled ID 0 (background). They may be smaller than a pixel.")

# ---------- POWER-4 CONNECTIVITY (print per catchment, fresh run, flush every=1) ----------
for lambda_parameter in [5]:
    out_csv = output_dir / f"lambda_{lambda_parameter}_catchment_connectivity.csv"
    # Fresh start: if a CSV exists from a previous run, overwrite it
    if out_csv.exists():
        out_csv.unlink()
    write_header = True  # write header only for the first row we append

    with rasterio.open(zeros_resampled_file) as rz, rasterio.open(ones_resampled_file) as ro:
        done, total = 0, len(order)
        for cid in order:
            # build a tight crop window lazily for this ID only
            rows, cols = np.where(catchment_raster_upd == cid)
            if rows.size == 0:
                continue
            r0, r1 = rows.min(), rows.max() + 1
            c0, c1 = cols.min(), cols.max() + 1
            sl_r, sl_c = slice(r0, r1), slice(c0, c1)

            print(f"→ λ={lambda_parameter} starting cid={cid} (px={counts[cid]}, "
                  f"win={r1-r0}×{c1-c0})", flush=True)

            mask = (catchment_raster_upd[sl_r, sl_c] == cid).astype(np.uint8, copy=False)
            if mask.sum() == 0:
                continue

            # crop arrays already in memory
            base = baseline_condition_raster_power4_upd[sl_r, sl_c]
            aff  = afforested_condition_raster_power4_upd[sl_r, sl_c]

            # lazily read just this window from zeros/ones on disk
            win = rasterio.windows.Window.from_slices(sl_r, sl_c)
            zeros_ = rz.read(1, window=win)
            ones_  = ro.read(1, window=win)

            baseline_conn_power4 = connectivity.landscape_connectivity(
                base, n_processes, mask, lambda_parameter, generations_mode, number_of_species_generations
            )
            aff_conn_power4 = connectivity.landscape_connectivity(
                aff,  n_processes, mask, lambda_parameter, generations_mode, number_of_species_generations
            )
            test_zeros_conn = connectivity.landscape_connectivity(
                zeros_, n_processes, mask, lambda_parameter, generations_mode, number_of_species_generations
            )
            test_ones_conn  = connectivity.landscape_connectivity(
                ones_,  n_processes, mask, lambda_parameter, generations_mode, number_of_species_generations
            )

            perc_diff_power4 = _pct_diff(aff_conn_power4, baseline_conn_power4)
            baseline_norm_p4 = _norm_try(baseline_conn_power4, test_ones_conn, test_zeros_conn)
            aff_norm_p4      = _norm_try(aff_conn_power4,      test_ones_conn, test_zeros_conn)

            # print per-catchment connectivity line
            print(
                f"[λ={lambda_parameter}] Catchment {cid}: "
                f"baseline_p4={baseline_conn_power4:.2f}, baseline_norm_p4={baseline_norm_p4:.3f}, "
                f"afforested_p4={aff_conn_power4:.2f}, diff_p4={perc_diff_power4:.1f}%, "
                f"aff_norm_p4={aff_norm_p4:.3f}",
                flush=True
            )

            # write back to GeoDataFrame
            catchments.loc[catchments["catchment_uid"] == cid, out_cols] = [
                baseline_conn_power4, baseline_norm_p4, aff_conn_power4, perc_diff_power4, aff_norm_p4
            ]

            # append one row per catchment to CSV (fresh file)
            row = {
                "catchment_uid": cid,
                "baseline_catchment_connectivity_power4": baseline_conn_power4,
                "baseline_normalized_power4": baseline_norm_p4,
                "afforested_catchment_connectivity_power4": aff_conn_power4,
                "percentage_difference_catchment_connectivity_power4": perc_diff_power4,
                "afforested_normalized_power4": aff_norm_p4,
            }
            pd.DataFrame([row]).to_csv(out_csv, mode="a", header=write_header, index=False)
            write_header = False  # only for the very first append

            done += 1
            if (done % 5) == 0 or done == total:
                print(f"Progress λ={lambda_parameter}: {done}/{total}", flush=True)

    print(f"Done with λ={lambda_parameter}. Appended rows to {out_csv}")

    # Save WITH geometry as a GeoPackage (recommended)
    gpkg_path = output_dir / f"lambda_{lambda_parameter}_catchments_power4.gpkg"
    catchments.to_file(gpkg_path, driver="GPKG", layer="catchments")
    print(f"Also wrote GeoPackage (with geometry) → {gpkg_path}")

In [ ]:
# ---------- Cartographic helpers (as before) ----------
def add_scale_bar(ax, length_km=20, location=(0.9, 0.79), fontsize=14, lw=3):
    half = 0.05; x,y = location
    ax.plot([x-half, x+half], [y, y], transform=ax.transAxes, color='black', lw=2)
    for pos in (x-half, x, x+half):
        ax.plot([pos, pos], [y-0.005, y+0.005], transform=ax.transAxes, color='black', lw=2)
    ax.text(x-half, y-0.03, "0", transform=ax.transAxes, ha='center', va='center')
    ax.text(x,       y-0.03, f"{int(length_km//2)}", transform=ax.transAxes, ha='center', va='center')
    ax.text(x+half,  y-0.03, f"{int(length_km)}",    transform=ax.transAxes, ha='center', va='center')
    ax.text(x+half+0.02, y, "km", transform=ax.transAxes, ha='left', va='center')

def add_north_arrow(ax, location=(0.9, 0.85), size=0.05, fontsize=14, lw=3):
    x,y = location
    ax.annotate(
        "", xy=(x, y+size), xycoords='axes fraction',
        xytext=(x, y), textcoords='axes fraction',
        arrowprops=dict(facecolor='black', edgecolor='black',
                        headwidth=10, headlength=15, width=5)
    )
    ax.text(x, y+size+0.02, "N", transform=ax.transAxes,
            ha='center', va='center', fontsize=12, fontweight='bold')

# ---------- Load GeoDataFrame with metrics ----------
# Use the in-memory 'catchments' if it already has the columns; otherwise load/merge.
needed = {
    "baseline_normalized_power4",
    "afforested_normalized_power4",
    "percentage_difference_catchment_connectivity_power4",
    "catchment_uid",
}
def _ensure_metrics(catchments):
    if needed.issubset(set(catchments.columns)):
        gdf = catchments.copy()
    else:
        # Try to load from the GPKG saved by your loop:
        if gpkg_path.exists():
            gdf = gpd.read_file(gpkg_path, layer="catchments")
        else:
            # Fallback: merge from the CSV onto your existing catchments polygons
            gdf = catchments.copy()
            conn = pd.read_csv(csv_path)
            gdf = gdf.merge(conn, on="catchment_uid", how="left")
    # drop background/invalid
    if "catchment_uid" in gdf.columns:
        gdf = gdf.loc[gdf["catchment_uid"].fillna(0).astype(int) > 0].copy()
    return gdf

gdf = _ensure_metrics(catchments)

label_map = {
    "baseline_normalized_power4": "Baseline normalized connectivity (power 4)",
    "afforested_normalized_power4": "Afforested normalized connectivity (power 4)",
    "percentage_difference_catchment_connectivity_power4": "Change in connectivity (%)",
}

def _smart_norm_cmap(values, mode):
    vals = np.asarray(values, dtype=float)
    vals = vals[~np.isnan(vals)]
    if vals.size == 0:
        # safe fallback
        return mpl.colors.Normalize(vmin=0, vmax=1), plt.colormaps["Greens"]
    if mode == "diverging":  # for % difference
        a = float(np.nanmax(np.abs(vals)))
        a = 1.0 if a == 0 else a
        norm = mpl.colors.TwoSlopeNorm(vmin=-a, vcenter=0.0, vmax=a)
        cmap = plt.colormaps["RdBu_r"]
    else:  # baseline/afforested normalized → [0, max]
        vmax = float(np.nanmax(vals))
        vmax = 1.0 if vmax == 0 else vmax
        norm = mpl.colors.Normalize(vmin=0.0, vmax=vmax)
        cmap = plt.colormaps["Greens"]
    return norm, cmap

def plot_connectivity_map(gdf, column, title, fname, diverging=False, label_ids=True):
    fig, ax = plt.subplots(figsize=(18, 14), dpi=300)

    mode = "diverging" if diverging else "sequential"
    norm, cmap = _smart_norm_cmap(gdf[column], mode)

    # fill polygons
    gdf.plot(ax=ax, column=column, cmap=cmap, norm=norm, linewidth=0, alpha=0.85)

    # boundaries
    gdf.boundary.plot(ax=ax, edgecolor='black', linewidth=0.5)

    # continuous colorbar
    sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])  # required for colorbar
    cbar = fig.colorbar(sm, ax=ax, fraction=0.03, pad=0.04)
    cbar.set_label(label_map.get(column, column), family='Times New Roman')

    # labels
    if label_ids and "catchment_uid" in gdf.columns:
        for _, row in gdf.iterrows():
            pt = row.geometry.representative_point()
            ax.text(pt.x, pt.y, str(int(row["catchment_uid"])),
                    ha='center', va='center', fontsize=6, fontweight='bold')

    # legend stub for boundaries
    handle = Line2D([0], [0], color='black', lw=1.5, label='Catchment boundary')
    leg = ax.legend(handles=[handle], title='Legend',
                    bbox_to_anchor=(0.5, -0.15), loc='upper center',
                    frameon=False, fontsize=20, title_fontsize=16,
                    prop={'family':'Times New Roman'})
    leg.get_title().set_position((0, 10))

    add_scale_bar(ax)
    add_north_arrow(ax)

    ax.set_axis_off()
    plt.title(title, fontsize=20, fontweight='bold',
              fontname='Times New Roman', pad=20)
    plt.tight_layout()

    out_png = Path(output_dir) / f"{fname}.png"
    out_pdf = Path(output_dir) / f"{fname}.pdf"
    fig.savefig(out_png, dpi=300, bbox_inches='tight')
    fig.savefig(out_pdf,              bbox_inches='tight')
    print(f"Saved {out_png} and {out_pdf}")
    plt.show()

# ---------- Make the three figures ----------
plot_connectivity_map(
    gdf,
    column="baseline_normalized_power4",
    title="Baseline Normalized Connectivity (Power 4)\nby Catchment in Jamaica",
    fname="Figure_baseline_connectivity_power4",
    diverging=False
)

plot_connectivity_map(
    gdf,
    column="afforested_normalized_power4",
    title="Afforested Normalized Connectivity (Power 4)\nby Catchment in Jamaica",
    fname="Figure_afforested_connectivity_power4",
    diverging=False
)

plot_connectivity_map(
    gdf,
    column="percentage_difference_catchment_connectivity_power4",
    title="Change in Connectivity After Afforestation (Power 4)\nby Catchment in Jamaica",
    fname="Figure_change_connectivity_power4",
    diverging=True
)

In [ ]:


# 1) Build a bounded, monotonic “increase” metric
b = gdf["baseline_normalized_power4"].astype(float).to_numpy()
a = gdf["afforested_normalized_power4"].astype(float).to_numpy()

b = np.clip(b, 0.0, 1.0)
a = np.clip(a, 0.0, 1.0)

delta_pp = (a - b) * 100.0
# enforce your expectation (no decreases); treat tiny negatives as numeric noise
tol = 1e-9
delta_pp = np.where(delta_pp < tol, 0.0, delta_pp)
delta_pp = np.clip(delta_pp, 0.0, 100.0)

gdf["delta_normalized_power4_pct"] = delta_pp

# 2) Replot with a 0–100 sequential scale (no diverging colormap)
plot_connectivity_map(
    gdf,
    column="delta_normalized_power4_pct",
    title="Increase in Normalized Connectivity (Power 4)\nby Catchment in Jamaica",
    fname="Figure_change_connectivity_power4",   # overwrite if you like
    diverging=False
)

In [ ]:


def _to_percent(arr):
    a = np.asarray(arr, float)
    if np.nanmax(np.abs(a)) <= 1.000001:  # if in [0,1], convert to %
        a = a * 100.0
    return a

# 1) Prepare clean baseline/afforested in PERCENT (0–100)
b = _to_percent(gdf["baseline_normalized_power4"])
a = _to_percent(gdf["afforested_normalized_power4"])

# clamp to valid range and handle NaNs
b = np.clip(b, 0.0, 100.0)
a = np.clip(a, 0.0, 100.0)

# 2) Percentage-point gain (aff − base), enforce non-negative, cap at 100
tol = 1e-9
delta_pp = a - b
delta_pp = np.where(delta_pp < tol, 0.0, delta_pp)
delta_pp = np.clip(delta_pp, 0.0, 100.0)

gdf["delta_normalized_power4_pp"] = delta_pp

# 3) Replot with fixed 0–100 scale so the legend is interpretable
def plot_pp_gain(gdf, column, title, fname):
    fig, ax = plt.subplots(figsize=(18,14), dpi=300)
    cmap = plt.colormaps["Greens"]
    norm = mpl.colors.Normalize(vmin=0, vmax=100)

    gdf.plot(
        ax=ax, column=column, cmap=cmap, norm=norm, linewidth=0, alpha=0.85,
        missing_kwds={"color":"lightgrey","hatch":"///","label":"No data"}
    )
    gdf.boundary.plot(ax=ax, edgecolor="black", linewidth=0.5)

    sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.03, pad=0.04)
    cbar.set_label("Increase in normalized connectivity (pp)", family="Times New Roman")
    cbar.set_ticks([0, 25, 50, 75, 100])

    # label catchments if available
    if "catchment_uid" in gdf.columns:
        for _, row in gdf.iterrows():
            pt = row.geometry.representative_point()
            ax.text(pt.x, pt.y, str(int(row["catchment_uid"])),
                    ha='center', va='center', fontsize=6, fontweight='bold')

    handle = Line2D([0],[0], color='black', lw=1.5, label='Catchment boundary')
    leg = ax.legend(handles=[handle], title='Legend',
                    bbox_to_anchor=(0.5, -0.15), loc='upper center',
                    frameon=False, fontsize=20, title_fontsize=16,
                    prop={'family':'Times New Roman'})
    leg.get_title().set_position((0, 10))

    add_scale_bar(ax); add_north_arrow(ax)
    ax.set_axis_off()
    plt.title(title, fontsize=20, fontweight='bold', fontname='Times New Roman', pad=20)
    plt.tight_layout()

    out_png = Path(output_dir)/f"{fname}.png"
    out_pdf = Path(output_dir)/f"{fname}.pdf"
    fig.savefig(out_png, dpi=300, bbox_inches='tight')
    fig.savefig(out_pdf,              bbox_inches='tight')
    print(f"Saved {out_png} and {out_pdf}")
    plt.show()

plot_pp_gain(
    gdf,
    column="delta_normalized_power4_pp",
    title="Increase in Normalized Connectivity (Power 4) — Percentage Points (0–100)",
    fname="Figure_change_connectivity_power4_pp"
)

# Optional: if you still want a % of baseline map but capped 0–100:
base_raw = gdf["baseline_catchment_connectivity_power4"].astype(float).to_numpy()
aff_raw  = gdf["afforested_catchment_connectivity_power4"].astype(float).to_numpy()
eps = 1e-9
delta_pct_clamped = (aff_raw - base_raw) / np.maximum(base_raw, eps) * 100.0
delta_pct_clamped = np.clip(delta_pct_clamped, 0.0, 100.0)
gdf["delta_connectivity_pct_clamped"] = delta_pct_clamped